# init

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from nsqdriver import MCIDriver, QSYNCDriver
from nsqdriver.NS_MCI import SHARED_DEVICE_MEM
import nsqdriver.nswave as nw

SHARED_DEVICE_MEM.clear_ip()
plt.style.use('ggplot') 

# ---------- 1.1 Device address ----------
deviceIP = "172.16.153.203"

# ---------- 1.2 Sample-rate configuration ----------
da_rate_rd = 8e9     # readout pulse (probe / readout)
da_rate_xy = 8e9     # XY drive
da_rate_z  = 4e9     # Z bias
ad_rate    = 4e9     # AD capture

# ---------- 1.3 Channel - sample-rate mapping ----------
da_channel_rate = {
    "S2-O1":  da_rate_rd,    # readout channel
    "S13-O1": da_rate_xy,    # XY drive channel
    "S12-O1": da_rate_z,     # Z bias channel
}

# ---------- 1.4 System parameters ----------
sysparam = {
    "MixMode":   1,          # DA mix mode (1: first Nyquist zone, 2: second Nyquist zone)
    "INMixMode": 2,          # AD mix mode
    "RefClock":  "out",      # MCI outputs the reference clock
    "ADrate":    ad_rate,
    **{f"DArate_{ch}": rate for ch, rate in da_channel_rate.items()},
}

qsync_param = {
    "TrigFrom": 0,           # trigger source
    "RefClock": "in",        # QSYNC receives external reference clock (from MCI)
}

# ---------- 1.5 Device instantiation and synchronization ----------
device = MCIDriver(deviceIP, 40)
qsync  = QSYNCDriver(deviceIP)

qsync.open(system_parameter=qsync_param)     # open QSYNC first (provides clock reference)
device.open(system_parameter=sysparam)       # then open MCI
qsync.sync_system()                          # system synchronization alignment
time.sleep(1)

*********QSYNC172.16.153.203开启成功*********
core_temp: 50℃
10M_locked: 0
qsync 172.16.153.203 opened successfully
*********设备172.16.153.203开启成功*********
device_type: pqtm
backend_version: v2.11.7-17-g54e06cd
ad_num: 4
da_num: 12
cpu_temp: nan
memory_use: 7.6
available chnl: 
OUT Chnl: ['S2-O1', 'S2-O2', 'S2-O3', 'S2-O4', 'S13-O1', 'S13-O2', 'S13-O3', 'S13-O4', 'S13-O5', 'S13-O6', 'S13-O7', 'S13-O8'] 

IN  Chnl: ['S2-I1', 'S2-I2', 'S2-I3', 'S2-I4']
172.16.153.203开启成功
System synchronization succeeded


# np.ndarray

In [3]:
@nw.kernel
def program_test(wave_list:nw.Var):
    srate: nw.Var = 8e9
    time_width: nw.Var = 2e-6
    freq: nw.Var = 100e6
    time_line: np.ndarray = np.linspace(0, time_width, int(time_width * srate), endpoint=False)
    wave: np.ndarray = np.sin(2 * np.pi * time_line * freq)
    
    for wave in range(4):
        nw.capture(1e-6, 0, 0)
        nw.wait(1e-6)
    return nw.Kernel()
_k = program_test()


In [4]:
device.set('ProgramOUT', _k, f'S13-O1')

In [5]:
_k.parse()
a, b, c, d, e= nw.ir_pass(_k)


In [6]:
print(e)

capi 0b11111111 250 0 0 0b0111        # seq idx tag 0
witi 245                              # seq idx tag 1
nop                                   # seq idx tag 2
nop                                   # seq idx tag 3
nop                                   # seq idx tag 4
witi 244                              # seq idx tag 5
nop                                   # seq idx tag 6
nop                                   # seq idx tag 7
nop                                   # seq idx tag 8
capi 0b11111111 250 0 0 0b0111        # seq idx tag 9
witi 245                              # seq idx tag 10
nop                                   # seq idx tag 11
nop                                   # seq idx tag 12
nop                                   # seq idx tag 13
witi 244                              # seq idx tag 14
nop                                   # seq idx tag 15
nop                                   # seq idx tag 16
nop                                   # seq idx tag 17
capi 0b11111111 250 

# nw.Envelope

In [7]:
@nw.kernel
def program(wave_list: nw.Var):
    srate: nw.Var = 8e9
    i: nw.Var = 0
    time_width: nw.Var = 2e-6
    freq: nw.Var = 100e6
    time_line_1: np.ndarray = np.linspace(0, time_width, int(time_width * srate), endpoint=False)
    wave: np.ndarray = np.cos(2 * np.pi * time_line_1 * freq)
    envelope: nw.Envelope
    nw.init_frame(0, 0.5 * np.pi)
    nw.wait_for_trigger()
    for wave in wave_list:
        envelope = nw.ins_envelope(wave)
        nw.play_wave(envelope, 1, 0, 0)
        nw.wait(2.8e-6)

    return nw.Kernel()

da_srate = 8e9
time_width = 1e-6
freq_1 = 4.1e9
freq_2 = 4.11e9
freq_3 = 4.12e9
time_line = np.linspace(0, time_width, round(da_srate * time_width), endpoint=False)
wave_list = [np.cos(time_line * freq_1), np.cos(time_line * freq_2), np.cos(time_line *freq_3)]

_k = program(wave_list)


In [8]:
device.set('ProgramOUT', _k, f'S13-O1')

In [9]:
_k.parse()
_, _, _, _, asm= nw.ir_pass(_k)
print(e)

capi 0b11111111 250 0 0 0b0111        # seq idx tag 0
witi 245                              # seq idx tag 1
nop                                   # seq idx tag 2
nop                                   # seq idx tag 3
nop                                   # seq idx tag 4
witi 244                              # seq idx tag 5
nop                                   # seq idx tag 6
nop                                   # seq idx tag 7
nop                                   # seq idx tag 8
capi 0b11111111 250 0 0 0b0111        # seq idx tag 9
witi 245                              # seq idx tag 10
nop                                   # seq idx tag 11
nop                                   # seq idx tag 12
nop                                   # seq idx tag 13
witi 244                              # seq idx tag 14
nop                                   # seq idx tag 15
nop                                   # seq idx tag 16
nop                                   # seq idx tag 17
capi 0b11111111 250 

# nw.frame

In [7]:
@nw.kernel
def program():
    judge_reg0: nw.Reg = 0
    srate: nw.Var = 8e9
    time_width: nw.Var = 2e-6
    freq: nw.Var = 100e6
    time_line_1: np.ndarray =np.linspace(0, time_width, int(time_width * srate),endpoint=False)
    wave: np.ndarray = np.cos(2 * np.pi * time_line_1 * freq)
    envelope: nw.Envelope = nw.ins_envelope(wave)
    frame_0: nw.Frame = nw.init_frame(0, 0.5 * np.pi)
    frame_1: nw.Frame = nw.init_frame(100e6, 0.5 * np.pi)
    nw.wait_for_trigger()
    nw.play_wave(envelope, 1, 0, 0)
    
    return nw.Kernel()


In [8]:
_k = program()

In [39]:
device.set('ProgramOUT', _k, f'S13-O1')

# Loop

In [5]:
@nw.kernel
def  program(param: nw.Var):
    nw.wait_for_trigger()
    i: nw.Var
    for i in param:
        nw.wait(400e-9)
        nw.capture(2e-6, 160e-9, 2e-6)
    return nw.Kernel()

param = np.arange(0, 10)
_k = program(param)

In [6]:
device.set('ProgramOUT', _k, f'S13-O1')

In [47]:
_k.parse()
_, _, _, _, asm= nw.ir_pass(_k)
print(asm)

wtg                                      # seq idx tag 0
nop                                      # seq idx tag 1
nop                                      # seq idx tag 2
nop                                      # seq idx tag 3
witi 94                                  # seq idx tag 4
nop                                      # seq idx tag 5
nop                                      # seq idx tag 6
nop                                      # seq idx tag 7
capi 0b11111111 500 40 500 0b0111        # seq idx tag 8
witi 535                                 # seq idx tag 9
nop                                      # seq idx tag 10
nop                                      # seq idx tag 11
nop                                      # seq idx tag 12
witi 94                                  # seq idx tag 13
nop                                      # seq idx tag 14
nop                                      # seq idx tag 15
nop                                      # seq idx tag 16
capi 0b11111111 500 40 5

In [60]:
import nsqdriver
print(nsqdriver.__version__)

0.12.18


In [14]:
@nw.kernel
def program():

    judge_reg_1: nw.Reg = 0
    i: nw.Reg = 0
    ii: nw.Reg = 0
    srate: nw.Var = 8e9
    time_width: nw.Var = 1.024e-6
    freq: nw.Var = 100e6
    time_line: np.ndarray = np.linspace(0, time_width, int(time_width * srate))
    wave: np.ndarray = np.cos(2 * np.pi * time_line * freq)
    frame_0: nw.Frame = nw.init_frame(0, 0.5*np.pi)
    envelope_0: nw.Envelope = nw.ins_envelope(wave)

    nw.wait_for_trigger()
    for ii in nw.normalloop(2):
         nw.play_wave(envelope_0)
         nw.play_wave(envelope_0)
        # nw.play_wave(envelope_0)

    return nw.Kernel()



In [15]:
_k = program()
_k.parse()
_, _, _, _, asm= nw.ir_pass(_k)
print(asm)

movi %A0 0                                                # seq idx tag 0
movi %A1 0                                                # seq idx tag 1
movi %A2 0                                                # seq idx tag 2
fmsi 0b00000000000000000000000000000001 0 33554432        # seq idx tag 3
nop                                                       # seq idx tag 4
nop                                                       # seq idx tag 5
nop                                                       # seq idx tag 6
nop                                                       # seq idx tag 7
wtg                                                       # seq idx tag 8
nop                                                       # seq idx tag 9
nop                                                       # seq idx tag 10
nop                                                       # seq idx tag 11
plyi 0 0 256 32767 0 0 0                                  # seq idx tag 12
witi 249                           

In [4]:
device.set('ProgramOUT', _k, f'S13-O1')

# branch

In [11]:
@nw.kernel
def program(param: nw.Var):
    srate: nw.Var = 8e9
    time_width: nw.Var = 2e-6
    freq_1: nw.Var = 400e6
    freq_2: nw.Var = 100e6
    time_line: np.ndarray = np.linspace(0, time_width, int(time_width * srate), endpoint=False)
    wave_1: np.ndarray = np.cos(2 * np.pi * time_line * freq_1)
    wave_2: np.ndarray = np.cos(2 * np.pi * time_line * freq_1)
    frame_0: nw.Frame = nw.init_frame(0, 0.5*np.pi)
    envelope_1: nw.Envelope = nw.ins_envelope(wave_1)
    envelope_2: nw.Envelope = nw.ins_envelope(wave_2) 
    
    nw.wait_for_trigger()
    if param==freq_1:
        nw.play_wave(envelope_1)
    else:
        nw.play_wave(envelope_2)
    
    return nw.Kernel()


In [12]:
_k = program(400e6)
_k.parse()
_, _, _, _, asm= nw.ir_pass(_k)
print(asm)
 # = nw.Abi.compile(asm)

fmsi 0b00000000000000000000000000000001 0 33554432        # seq idx tag 0
nop                                                       # seq idx tag 1
nop                                                       # seq idx tag 2
nop                                                       # seq idx tag 3
nop                                                       # seq idx tag 4
wtg                                                       # seq idx tag 5
nop                                                       # seq idx tag 6
nop                                                       # seq idx tag 7
nop                                                       # seq idx tag 8
jali $0                                                   # seq idx tag 9
nop                                                       # seq idx tag 10
nop                                                       # seq idx tag 11
nop                                                       # seq idx tag 12
